In [1]:
import re

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

In [2]:
player_lookup = pd.read_csv("02_player_lookup.csv", index_col="player_id")["safe_name"]

In [3]:
driver = webdriver.Chrome()
try:
    driver.get("https://wc2026-tipping-nu.vercel.app/leaderboard")
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "lb-row")))

    points = []

    rows = driver.find_elements(by=By.CLASS_NAME, value="lb-row")

    for row in rows:
        row_button = row.find_element(by=By.TAG_NAME, value="button")
        on_click = row_button.get_attribute("onClick")
        player_id = re.match(R"viewPlayerTips\('(.+)'\)", on_click).group(1)
        points_element = row.find_element(by=By.CLASS_NAME, value="lb-pts")
        points.append({"player_id": player_id, "points": int(points_element.text)})
finally:
    driver.quit()

In [4]:
points_df = pd.DataFrame(points)
points_df["name"] = points_df["player_id"].map(player_lookup)
points_df.set_index("name")["points"].to_csv("04_points.csv")